In [371]:
import pandas as pd

from DWG_to_Excel_ETL import *
import os
import shutil
import numpy as np
import math

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\reask"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\reask\temp"
output_folder = r"C:\Users\Usuario\Desktop\projetos\reask\output"

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Reask - Eletromecânico.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Reask - Eletromecânico.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Reask - Eletromecânico.dwg")

data = extract_data_from_dxf(dxf_file)
print("Data Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Data Loaded


In [372]:
ELM_df = pd.DataFrame(data)

print(ELM_df.describe())
print(ELM_df.info())
print(ELM_df['name'].value_counts())

                   X              Y    rotation
count     231.000000     231.000000  231.000000
mean   700223.147202  755837.821441  102.857143
std       183.833798     662.469346   89.270334
min    699965.481602  755249.635348    0.000000
25%    700067.916030  755281.135348    0.000000
50%    700150.203520  755312.635348  180.000000
75%    700401.229759  756620.279054  180.000000
max    700567.786140  756686.279054  180.000000
<class 'pandas.DataFrame'>
RangeIndex: 231 entries, 0 to 230
Data columns (total 17 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   type      231 non-null    str    
 1   layer     231 non-null    str    
 2   handle    231 non-null    str    
 3   name      231 non-null    str    
 4   X         231 non-null    float64
 5   Y         231 non-null    float64
 6   rotation  231 non-null    float64
 7   UFV       231 non-null    str    
 8   PST       231 non-null    str    
 9   INV       231 non-null    str    
 10  E

In [373]:
struct_mask = (ELM_df['name'] == "EMF422-1_string_tilt_15°")

struct_df = ELM_df[struct_mask].copy()
struct_df.rename(columns={'X': 'Position_X', 'Y': 'Position_Y'}, inplace=True)
struct_df.drop(columns=['rotation', 'type', 'handle', 'QBT', 'DSJ', 'PVM', 'VBT'], inplace=True)
struct_df.to_excel(r"C:\Users\Usuario\Desktop\teste2.xlsx")
print(struct_df)
print(struct_df.describe())
print(struct_df.info())

          layer                      name     Position_X     Position_Y UFV  \
0    EMF_PST_01  EMF422-1_string_tilt_15°  700355.177645  756609.279054       
1    EMF_PST_01  EMF422-1_string_tilt_15°  700366.897645  756609.279054       
2    EMF_PST_01  EMF422-1_string_tilt_15°  700378.617645  756609.279054       
3    EMF_PST_01  EMF422-1_string_tilt_15°  700390.337645  756609.279054       
4    EMF_PST_01  EMF422-1_string_tilt_15°  700402.057645  756609.279054       
..          ...                       ...            ...            ...  ..   
217  EMF_PST_02  EMF422-1_string_tilt_15°  700121.272377  755249.635348       
218  EMF_PST_02  EMF422-1_string_tilt_15°  700109.552377  755249.635348       
219  EMF_PST_02  EMF422-1_string_tilt_15°  700097.832377  755249.635348       
220  EMF_PST_02  EMF422-1_string_tilt_15°  700086.112377  755249.635348       
221  EMF_PST_02  EMF422-1_string_tilt_15°  700074.392377  755249.635348       

    PST INV EFX STR GRP  
0                        

In [374]:
inverter_mask = (ELM_df['name'] == "EMF422-1_inversor")

inverter_df = ELM_df[inverter_mask].copy()
inverter_df.rename(columns={'X': 'inverter_x', 'Y': 'inverter_y'}, inplace=True)
inverter_df.drop(columns=['name', 'rotation', 'type', 'handle', 'layer', 'EFX', 'STR'], inplace=True)
inverter_df.sort_values(by=['inverter_y', 'inverter_x'], ascending=[False, True], inplace=True)
print(inverter_df)

        inverter_x     inverter_y UFV PST INV GRP VBT QBT DSJ PVM
222  700440.045418  756661.203052  01  01  01                    
223  700506.892574  756638.878795  01  01  02                    
224  700542.413435  756625.274054  01  01  03                    
225  700567.786140  756614.274054  01  01  04                    
226  699969.450403  755317.130348  02  01  01                    
227  699966.552097  755296.130348  02  01  02                    
228  699994.257602  755275.130348  02  01  03                    
229  700037.591780  755264.630348  02  01  04                    
230  700071.013721  755253.620612  02  01  05                    


In [375]:
struct_df['GRP'] = struct_df["UFV"].astype(str) + "." + struct_df["PST"].astype(str) + '.' + struct_df['INV'].astype(str).str.zfill(2)
inverter_df['GRP'] = inverter_df["UFV"].astype(str) + "." + inverter_df["PST"].astype(str) + '.' + inverter_df['INV'].astype('Int64').astype(str).str.zfill(2)

print(struct_df)
print(inverter_df)

          layer                      name     Position_X     Position_Y UFV  \
0    EMF_PST_01  EMF422-1_string_tilt_15°  700355.177645  756609.279054       
1    EMF_PST_01  EMF422-1_string_tilt_15°  700366.897645  756609.279054       
2    EMF_PST_01  EMF422-1_string_tilt_15°  700378.617645  756609.279054       
3    EMF_PST_01  EMF422-1_string_tilt_15°  700390.337645  756609.279054       
4    EMF_PST_01  EMF422-1_string_tilt_15°  700402.057645  756609.279054       
..          ...                       ...            ...            ...  ..   
217  EMF_PST_02  EMF422-1_string_tilt_15°  700121.272377  755249.635348       
218  EMF_PST_02  EMF422-1_string_tilt_15°  700109.552377  755249.635348       
219  EMF_PST_02  EMF422-1_string_tilt_15°  700097.832377  755249.635348       
220  EMF_PST_02  EMF422-1_string_tilt_15°  700086.112377  755249.635348       
221  EMF_PST_02  EMF422-1_string_tilt_15°  700074.392377  755249.635348       

    PST INV EFX STR   GRP  
0                    ..

In [376]:
struct_df['Comparing X'] = struct_df.groupby('GRP')['Position_X'].transform('mean')

struct_df = struct_df.merge(inverter_df, on='GRP')

struct_df['E or W'] = np.where(
    struct_df['inverter_x'] >= struct_df['Comparing X'],
    'E',
    'W'
)

struct_df.sort_values(by=['UFV_x', 'PST_x', 'INV_x', 'Position_Y', 'Position_X'], ascending=[True, True, True, False, True], inplace=True)

struct_df["STR"] = struct_df.groupby("GRP").cumcount()+1
struct_df["TAG"] = struct_df['GRP'] + "." + struct_df['STR'].astype(str).str.zfill(2)


print(struct_df)
print(struct_df["STR"])

Empty DataFrame
Columns: [layer, name, Position_X, Position_Y, UFV_x, PST_x, INV_x, EFX, STR, GRP, Comparing X, inverter_x, inverter_y, UFV_y, PST_y, INV_y, VBT, QBT, DSJ, PVM, E or W, TAG]
Index: []

[0 rows x 22 columns]
Series([], Name: STR, dtype: int64)


In [377]:
def group_structures(df, inverter_position, max_distance=12):
    """
    Groups adjacent structures in trios (3) while respecting:
    - TAG boundaries
    - Maximum distance constraint (22m default)
    - Sorted by Position Y then X
    """
    # Sort by Position X (primary) and Position Y (secondary)

    df = df.sort_values(['Position_Y', 'Position_X'], ascending=[False, True]).reset_index(drop=True)

    # Calculate pairwise distances between consecutive structures
    df['x_diff'] = round(df['Position_X'].diff(),2)
    df['y_diff'] = round(df['Position_Y'].diff(),2)
    df['distance'] = np.sqrt(df['x_diff']**2 + df['y_diff']**2)

    # Create group boundaries where:
    # 1. Y is different OR
    # 2. Distance > max_distance OR
    # 3. It's the first structure
    df['group_boundary'] = (
        (abs(df['y_diff']) > 0.1) |
        (abs(df['x_diff']) > max_distance) |
        (df.index == 0)
    )
    # Create group IDs
    df['group_id'] = df['group_boundary'].cumsum()

    # Split into groups
    result_groups = []

    for _, group_df in df.groupby('group_id'):
        if inverter_position == "W":
            group_df = group_df.iloc[::-1].reset_index(drop=True)
        # Split groups larger than 3 into trios
        size = len(group_df)
        if size <= 3:
            result_groups.append(group_df)

            continue

        n_trios = size // 3
        for i in range(n_trios):
            result_groups.append(group_df.iloc[i*3:i*3+3])


        if size % 3:  # Handle groups that have sizes that are not divisible by 3
            if (size % 3) % 2:
                result_groups.append(group_df.iloc[-1:])
            else:
                result_groups.append(group_df.iloc[-2:])

    if inverter_position == "W":
        result_groups = result_groups[::-1]
    df.to_excel(r"C:\Users\Usuario\Desktop\teste1.xlsx")
    return result_groups

def create_group_positions(groups, inverter_position):
    """Create the output DataFrame from groups"""
    group_data = []
    for i, group_df in enumerate(groups, 1):
        if len(group_df) == 3:
            if inverter_position == "W":
                first_struct = group_df.iloc[2]
                second_struct = group_df.iloc[1]
                third_struct = group_df.iloc[0]
            else:
                first_struct = group_df.iloc[0]
                second_struct = group_df.iloc[1]
                third_struct = group_df.iloc[2]
            group_data.append({
                'Group_ID': i,
                'TAG-1': first_struct['TAG'],
                'TAG-2': second_struct['TAG'],
                'TAG-3': third_struct['TAG'],
                'Position_X': second_struct['Position_X'],
                'Position_Y': second_struct['Position_Y'],
                'Group_Size': len(group_df),
                'Layer-1': first_struct['layer'],
                'Layer-2': second_struct['layer'],
                'Layer-3': third_struct['layer']
            })
        elif len(group_df) == 2:
            if inverter_position == "W":
                first_struct = group_df.iloc[1]
                second_struct = group_df.iloc[0]
            else:
                first_struct = group_df.iloc[0]
                second_struct = group_df.iloc[1]
            group_data.append({
                'Group_ID': i,
                'TAG-1': first_struct['TAG'],
                'TAG-2': second_struct['TAG'],
                'Position_X': first_struct['Position_X'],
                'Position_Y': first_struct['Position_Y'],
                'Group_Size': len(group_df),
                'Layer-1': first_struct['layer'],
                'Layer-2': second_struct['layer']
            })
        else:
            first_struct = group_df.iloc[0]
            group_data.append({
                'Group_ID': i,
                'TAG-1': first_struct['TAG'],
                'Position_X': first_struct['Position_X'],
                'Position_Y': first_struct['Position_Y'],
                'Group_Size': len(group_df),
                'Layer-1': first_struct['layer']
            })
    return pd.DataFrame(group_data)

In [378]:
estruturas_E = struct_df[struct_df['E or W'] == 'E']
estruturas_W = struct_df[struct_df['E or W'] == 'W']

results_E = group_structures(estruturas_E, 'E')
results_W = group_structures(estruturas_W, 'W')
print(results_E)
print(results_W)

group_positions_E = create_group_positions(results_E, "E")
group_positions_W = create_group_positions(results_W, "W")

group_positions = pd.concat([group_positions_E, group_positions_W], ignore_index=True)

# DEBUG: Check what columns actually exist
print("Columns in group_positions after concat:")
print(group_positions.columns.tolist())
print("\nFirst 5 rows:")
print(group_positions.head())
print("\nData types:")
print(group_positions.dtypes)
print("\nUnique Group_Size values:")
print(group_positions['Group_Size'].unique())

group_positions["UFV-1"] = group_positions['TAG-1'].str[0:2]
group_positions['UFV-2'] = np.where(
    group_positions["Group_Size"] > 1,
    group_positions['TAG-2'].str[0:2],
    None
)
group_positions['UFV-3'] = np.where(
    group_positions["Group_Size"] > 2,
    group_positions['TAG-3'].str[0:2],
    None
)

group_positions["PST-1"] = group_positions['TAG-1'].str[3:5]
group_positions['PST-2'] = np.where(
    group_positions["Group_Size"] > 1,
    group_positions['TAG-2'].str[3:5],
    None
)
group_positions['PST-3'] = np.where(
    group_positions["Group_Size"] > 2,
    group_positions['TAG-3'].str[3:5],
    None
)

group_positions["INV-1"] = group_positions['TAG-1'].str[6:8]
group_positions['INV-2'] = np.where(
    group_positions["Group_Size"] > 1,
    group_positions['TAG-2'].str[6:8],
    None
)
group_positions['INV-3'] = np.where(
    group_positions["Group_Size"] > 2,
    group_positions['TAG-3'].str[6:8],
    None
)

group_positions["STR-1"] = group_positions['TAG-1'].str[-2:]
group_positions['STR-2'] = np.where(
    group_positions["Group_Size"] > 1,
    group_positions['TAG-2'].str[-2:],
    None
)
group_positions['STR-3'] = np.where(
    group_positions["Group_Size"] > 2,
    group_positions['TAG-3'].str[-2:],
    None
)

group_positions['GRP-1'] = group_positions['TAG-1'].str[:-3]
group_positions['GRP-2'] = np.where(
    group_positions["Group_Size"] > 1,
    group_positions['TAG-2'].str[:-3],
    None
)
group_positions['GRP-3'] = np.where(
    group_positions["Group_Size"] > 2,
    group_positions['TAG-3'].str[:-3],
    None
)

print(group_positions)
group_positions.to_excel(r"C:\Users\Usuario\Desktop\teste.xlsx")

[]
[]
Columns in group_positions after concat:
[]

First 5 rows:
Empty DataFrame
Columns: []
Index: []

Data types:
Series([], dtype: object)

Unique Group_Size values:


KeyError: 'Group_Size'

In [ ]:
correction_position_x_2str = 5.86
correction_position_y_3str = 2.3204

def generate_script(row):
    group_size = row['Group_Size']

    if group_size == 1:
        return (
            f'(command-s "._layer" "set" "{row["Layer-1"]}" "")\n'
            f'(command-s "._insert" "EMF422-1_string_{group_size}" '
            f'"{round(row["Position_X"],4)},{round(row["Position_Y"],4)}" 1 1 0 "{row["UFV-1"]}" "{row["PST-1"]}" "{row["INV-1"]}" "" "{row["STR-1"]}" "{row["GRP-1"]}")'
        )

    elif group_size == 2:
        if row["Layer-1"] == row["Layer-2"]:
            return (
                f'(command-s "._layer" "set" "{row["Layer-1"]}" "")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"]+correction_position_x_2str,4)},{round(row["Position_Y"],4)}" 1 1 0 "{row["UFV-1"]}" "{row["PST-1"]}" "{row["INV-1"]}" "" "{row["STR-1"]}" "{row["GRP-1"]}")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"]+correction_position_x_2str,4)},{round(row["Position_Y"],4)}" 1 1 180 "{row["UFV-2"]}" "{row["PST-2"]}" "{row["INV-2"]}" "" "{row["STR-2"]}" "{row["GRP-2"]}")'
            )
        else:
            return (
                f'(command-s "._layer" "set" "{row["Layer-1"]}" "")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"]+correction_position_x_2str,4)},{round(row["Position_Y"],4)}" 1 1 0 "{row["UFV-1"]}" "{row["PST-1"]}" "{row["INV-1"]}" "" "{row["STR-1"]}" "{row["GRP-1"]}")\n'
                f'(command-s "._layer" "set" "{row["Layer-2"]}" "")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"]+correction_position_x_2str,4)},{round(row["Position_Y"],4)}" 1 1 180 "{row["UFV-2"]}" "{row["PST-2"]}" "{row["INV-2"]}" "" "{row["STR-2"]}" "{row["GRP-2"]}")'
            )

    else:  # group_size == 3
        # Check if all layers are the same
        if row["Layer-1"] == row["Layer-2"] == row["Layer-3"]:
            return (
                f'(command-s "._layer" "set" "{row["Layer-1"]}" "")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"],4)},{round(row["Position_Y"]+correction_position_y_3str,4)}" 1 1 0 "{row["UFV-1"]}" "{row["PST-1"]}" "{row["INV-1"]}" "" "{row["STR-1"]}" "{row["GRP-1"]}")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"],4)},{round(row["Position_Y"],4)}" 1 1 0 "{row["UFV-2"]}" "{row["PST-2"]}" "{row["INV-2"]}" "" "{row["STR-2"]}" "{row["GRP-2"]}")\n'
                f'(command-s "._insert" "EMF422-1_string_{group_size}" '
                f'"{round(row["Position_X"],4)},{round(row["Position_Y"]-correction_position_y_3str,4)}" 1 1 0 "{row["UFV-3"]}" "{row["PST-3"]}" "{row["INV-3"]}" "" "{row["STR-3"]}" "{row["GRP-3"]}")'
            )
        else:
            script = f'(command-s "._layer" "set" "{row["Layer-1"]}" "")\n'
            script += f'(command-s "._insert" "EMF422-1_string_{group_size}" '
            script += f'"{round(row["Position_X"],4)},{round(row["Position_Y"]+correction_position_y_3str,4)}" 1 1 0 "{row["UFV-1"]}" "{row["PST-1"]}" "{row["INV-1"]}" "" "{row["STR-1"]}" "{row["GRP-1"]}")\n'

            if row["Layer-1"] != row["Layer-2"]:
                script += f'(command-s "._layer" "set" "{row["Layer-2"]}" "")\n'

            script += f'(command-s "._insert" "EMF422-1_string_{group_size}" '
            script += f'"{round(row["Position_X"],4)},{round(row["Position_Y"],4)}" 1 1 0 "{row["UFV-2"]}" "{row["PST-2"]}" "{row["INV-2"]}" "" "{row["STR-2"]}" "{row["GRP-2"]}")\n'

            if row["Layer-2"] != row["Layer-3"]:
                script += f'(command-s "._layer" "set" "{row["Layer-3"]}" "")\n'

            script += f'(command-s "._insert" "EMF422-1_string_{group_size}" '
            script += f'"{round(row["Position_X"],4)},{round(row["Position_Y"]-correction_position_y_3str,4)}" 1 1 0 "{row["UFV-3"]}" "{row["PST-3"]}" "{row["INV-3"]}" "" "{row["STR-3"]}" "{row["GRP-3"]}")'

            return script

# Apply the function to each row
group_positions['Script'] = group_positions.apply(generate_script, axis=1)

print(group_positions)

with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Separador de Strings\Separador de Strings - Toolestown.scr", 'w') as f:
    f.write('\n'.join(group_positions["Script"].astype(str)) + '\n')

group_positions.to_excel(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Separador de Strings\teste.xlsx")